# Re-analysis in Python of a systematic review and meta-analysis: Global prevalence of gaming disorder

By: Mariana Lopes Peixe, r1125391

Course: Programming for Humanities [F0BR0a] - KU Leuven

Academic year: 2025-2026 

## 1. Introduction

Gaming Disorder (GD) has gained significant clinical attention, culminating in its inclusion in the 11th revision of the International Classification of Diseases (ICD-11). In 2021, Stevens et al. published a systematic review and meta-analysis on the global prevalence of GD, utilizing the software Meta-Essentials to report a global pooled prevalence of 3.05%.

For a current Meta-Analysis academic, I did a data audit and re-analysis of the Stevens et al. (2021) dataset using R (via the `metafor` package). Due to corrections during the data extraction phase, the R-based re-analysis yielded a slightly higher global pooled prevalence of 3.15%. While working on that project, I got curious as to how that data analysis could be done using Python and if that would change the results.
As a result, the current project, developed for the "Programming for Humanities" course, aims to translate this into Python. By replicating the previous R scripts using Python's data science stack (`pandas`, `seaborn`, `statsmodels`), this project serves as a methodological comparison across three distinct analytical environments:
1. The original meta-analysis by Stevens et al. (Meta-Essentials).
2. The previous data audit and re-analysis (R).
3. The current computational translation (Python).

The objectives are to verify that Python can replicate the R-based mathematical findings, generate aditional comparative data visualizations (graphics), and conduct a sensitivity analysis restricted to representative samples (Type 1 studies).

**Methodological Note & References:**
The mathematical formulas utilized in this notebook for data transformation (logits and variance) are the ones used by Stevens et al. (2021). These formulas were previously coded in R and are now adapted for Python. To replicate the Random-Effects Model without R's `metafor` package, the official documentation for Python's `statsmodels.stats.meta_analysis` module was consulted, specifically utilizing the `combine_effects` function. Since we did not learn about the `statsmodels`in class, this was a process of trial and error.

Documentation for `statsmodels.stats.meta_analysis`: https://www.statsmodels.org/dev/generated/statsmodels.stats.meta_analysis.combine_effects.html 

To help edit the Markdown text this “cheat sheet” was used: https://www.ibm.com/docs/en/db2-event-store/2.0.0?topic=notebooks-markdown-jupyter-cheatsheet 


###1.1 Structure

Since this project utilizes data from an Excel, it was divided into 3 folders: data, where the excel is; notebook, where this ipynb is; and a venv environment alongside a requirements.txt, to ensure this project runs of different devices.

In [ ]:
# Verify the active Python interpreter path to make sure the notebook is running inside the isolated virtual environment
import sys
print(sys.executable)



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



## 2. Data Review and Audit

The dataset utilized in this project is the audited version generated during the previous R re-analysis, which corrected minor discrepancies found in the original Stevens et al. (2021) findings. The final sample is 225,926 participants across 53 studies. 

The dataset includes the total sample size (N), the prevalence percentage, and the absolute number of cases (participants diagnosed with GD) for each study. The following code loads this dataset.


In [ ]:
#Load the audited data for the main analysis
df= pd.read_excel('../data/GD_prev.xlsx', sheet_name='MyReview')

#Display initial rows to verify the data is loaded
df.head()

### 2.1 Study Characteristics

To understand the distribution of the data, descriptive statistics are extracted. This includes the mean, minimum, and maximum values for both the total number of participants and the absolute number of positive cases across all 53 studies.


In [ ]:
#Extracting descriptive statistics
#Utilizing f-string to better mix text and variables in Python
#And :.0f to round the numbers
print("- N Statistics")
print(f"Mean Participants: {df['N'].mean():.0f}")
print(f"Minimum Participants: {df['N'].min()}")
print(f"Maximum Participants: {df['N'].max()}")

print("- Cases Statistics")
print(f"Mean Cases: {df['Cases'].mean():.0f}")
print(f"Minimum Cases: {df['Cases'].min():.0f}")
print(f"Maximum Cases: {df['Cases'].max():.0f}")



### 2.2 Visualizing the Sample Distribution

To better visualize the variation of the included studies, the distribution of participants and the absolute number of cases per study are in a graphic below. This highlights the differences in sample sizes and impact scale across the literature.

In [ ]:
#Defining a wide figure size to ensure the 53 studies can be properly seen
plt.figure(figsize=(18, 6))

#Plotting the total participants per study to visualize sample size distribution, using seaborn
sns.barplot(x=df.index + 1, y='N', data=df)

plt.title('Figure 1: Number of Participants per Study')
plt.xlabel('Study Id')
plt.ylabel('Number of participants')

#Display the plot
plt.show()

In [ ]:
#Same steps as before, but for the cases distribution
plt.figure(figsize=(18, 6))

sns.barplot(x=df.index + 1, y='Cases', data=df)

plt.title('Figure 2: Number of Cases per Study')
plt.xlabel('Study Id')
plt.ylabel('Number of cases')

plt.show()

## 3. Data Transformation

Because the prevalence estimates of GD are often very small, analyzing them directly can artificially inflate the heterogeneity between studies. To prevent this, the prevalence percentages must be transformed into logits (log odds) befrore running the Random-Effects Model. 

The formulas used for the logit transformation (Effect Size) and the calculation of the Variance are:

- Logit: $ES_{k} = \ln\left(\frac{P_{k}}{1 - P_{k}}\right)$
- Variance: $var = \frac{1}{N \cdot P} + \frac{1}{N \cdot (1 - P)}$

(Note: These formulas were extracted from Stevens' et al. (2021) article and I asked for help of Google Gemini to be able to translate these formulas into Markdown text)


In [ ]:
#Proportions (0 to 1) are required for logit transformations, so its needed to convert the percentages
df['p'] = df['Prev_pct'] / 100

#Transforming prevalence into log odds
df['logit'] = np.log(df['p'] / (1 - df['p']))

#Calculating the variance for each study based on sample size and proportion
df['var'] = (1 / (df['N'] * df['p'])) + (1 / (df['N'] * (1 - df['p'])))

#Displaying the first few rows to verify the transformations
df[['Study', 'Prev_pct', 'p', 'logit', 'var']].head()

## 4. Overall Re-analysis (Random-Effects Model)

To synthesize the pooled prevalence across the 53 studies, a Random-Effects Model is applied. While the original paper utilized Meta-Essentials and the re-analysis utilized R, this project relies on Python's `statsmodels.stats.meta_analysis` module. 

Once the model calculates the pooled effect size in logits, an inverse logit transformation is used to convert the estimate back into a readable percentage.


In [ ]:
from statsmodels.stats.meta_analysis import combine_effects

#Run the Random-Effects Model using the dl estimator
meta_results = combine_effects (df['logit'], df['var'], method_re='dl', row_names=df['Study'])

#Extract the random effects result ('mean_effect_re') (which is in logits)
pooled_logits = meta_results.mean_effect_re

#Apply the inverse logit formula to convert the logit back into a percentage
pooled_prev_prctg = (np.exp(pooled_logits) / (1 + np.exp(pooled_logits))) * 100

print(f"Global Pooled Prevalence: {pooled_prev_prctg:.2f}%:")

### 4.1 Heterogeneity Analysis

To quantify the variance between the included studies, some heterogeneity statistics can be visualized here (utilizing information that the model applied in the previous section already generated). 

- Cochran's $Q$: Tests whether the variability across studies is greater than what would be expected by chance.
- $I^2$: Represents the percentage of total variation across studies that is due to true heterogeneity rather than chance.
- $T^2$: Estimates the absolute variance of the true effect sizes.


In [ ]:
#statsmodels calculates these automatically:
#they are only being extracted here to divide this project in different sections.
q = meta_results.q

#Note: statsmodels outputs I^2 as a proportion (0 to 1), so it is multiplied by 100 for a percentage.
i2 = meta_results.i2 * 100

tau2 = meta_results.tau2

print("- Heterogeneity Statistics")
print(f"Cochran's Q: {q:.2f}")
print(f"I^2: {i2:.2f}%")
print(f"T^2: {tau2}")

## 5. Sensitivity Analysis

The original Stevens et al. paper noted high heterogeneity, so, to assess whether this was driven by lower-quality sampling methodologies, a sensitivity analysis was done. This restricts the dataset strictly to Type 1 studies, which utilize representative sampling - Stevens et al. (2021) defines this as “Representative [age, gender, education, socio-economic status, urbanicity] sampling, stratified random or randomized cluster sampling; population or cohort registry”.

In [ ]:
#Isolate Type 1 studies to run a sensitivity analysis
df_type1 = df[df['Type'] == 1].copy()

#Run the exact same Random-Effects Model on this subgroup
meta_type1 = combine_effects(df_type1['logit'], df_type1['var'], method_re='dl')

#Convert the Type 1 logit result back into a percentage
pooled_logit_type1 = meta_type1.mean_effect_re
pooled_prev_type1 = (np.exp(pooled_logit_type1) / (1 + np.exp(pooled_logit_type1))) * 100

print(f"Type 1 Prevalence: {pooled_prev_type1:.2f}%")

### 5.1 Comparing the Results

Finally, the drop in prevalence when restricting the analysis to representative sampling can be better visualized in the following graphic.

In [ ]:
categories = ['All Studies (k=53)', 'Type 1 Studies (k=31)']
prevalences = [pooled_prev_prctg, pooled_prev_type1]

plt.figure(figsize= (8, 6))

#Plot the comparison, using seaborn
#Using colors that are color-blind friendly
ax = sns.barplot(x=categories, y=prevalences, hue=categories, palette=['darkblue', 'darkorange'])

plt.title('Figure 4: Pooled Prevalence (%) Comparison')
plt.ylabel('Pooled Prevalence (%)')
plt.ylim(2.5, 3.3) 


plt.show()

## 6. Conclusion

This translation demonstrated that Python's `statsmodels` can effectively replicate the core meta-analytic findings of previous R-based data audits. Based on the audited data from 53 studies – in which studies varied significantly in scale, from 271 to 15,168 participants - the global pooled prevalence of Gaming Disorder was 3.17%. The prevalence is higher than Stevens' et al. (2021) 3.05%, and can be explained by the minor changes made during the data autdit phase (check excel sheet "MyRev+comments" to see which data was altered).The 0.2% difference (R = 3.15%, python = 3.17%), can be due to how R and Python's `statsmodels`handle rounding – since the results are very close to each other. 

Another finding of this analysis is the extreme between-study variance - the heterogeneity statistics revealed a Cochran’s Q of 4077.32 and an $I^2$ of 98.72%, indicating that nearly all the variance across the literature is due to heterogeneity rather than sampling error. 

To determine if this heterogeneity was driven by sampling methodologies, a sensitivity analysis was conducted: when the dataset was restricted to Type 1 studies (those utilizing representative sampling), the pooled prevalence dropped to 2.86%. 

Ultimately, this discrepancy suggests that non-representative sampling may artificially inflate the reported prevalence of Gaming Disorder. Besides that, the massive $I^2$ percentage confirms that regardless of the statistical software used (Meta-Essentials, R, or Python), the literature on Gaming Disorder in 2021 was deeply inconsistent - this shows an urgent need for standardized diagnostic frameworks and uniform sampling methodologies in future behavioral addiction research.
